# Getting Started with Unified Privacy Pipeline

This notebook introduces the core concepts and basic usage of the Unified Privacy Pipeline.

## What is the Unified Privacy Pipeline?

The Unified Privacy Pipeline integrates three critical privacy-preserving technologies:

1. **Differential Privacy (DP)**: Provides mathematical privacy guarantees during training
2. **Machine Unlearning**: Enables models to "forget" specific data points
3. **Influence Functions**: Traces and quantifies the impact of training data on predictions

## Installation

First, ensure all dependencies are installed:

In [ ]:
# Install required packages (if needed)
# !pip install torch torchvision numpy scikit-learn matplotlib opacus

In [ ]:
import sys
import os

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

# Import pipeline components
from influence_functions.influence_computation import create_influence_computer
from machine_unlearning.unlearning_methods import create_unlearner, UnlearningConfig
from evaluation.privacy_metrics import MembershipInferenceAttack

print("✅ Imports successful!")

## 1. Basic Setup: Create a Simple Model and Dataset

In [ ]:
# Define a simple neural network
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=20, output_dim=2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

# Create synthetic data
def create_synthetic_dataset(n_samples=500, input_dim=10):
    X = torch.randn(n_samples, input_dim)
    y = torch.randint(0, 2, (n_samples,))
    dataset = TensorDataset(X, y)
    return DataLoader(dataset, batch_size=32, shuffle=True)

# Create datasets
train_loader = create_synthetic_dataset(500)
forget_loader = create_synthetic_dataset(100)  # Data to unlearn
retain_loader = create_synthetic_dataset(400)  # Data to keep
test_loader = create_synthetic_dataset(200)

print("✅ Model and datasets created!")

## 2. Train a Simple Model

In [ ]:
# Initialize model
model = SimpleClassifier()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Train the model
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(20):
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}")

print("✅ Model training complete!")

## 3. Machine Unlearning Example

In [ ]:
# Create unlearning configuration
config = UnlearningConfig(
    learning_rate=0.01,
    max_iterations=20,
    patience=5
)

# Create gradient ascent unlearner
unlearner = create_unlearner("gradient_ascent", config)

# Perform unlearning
print("Starting unlearning process...")
results = unlearner.unlearn(model, forget_loader, retain_loader, device)

# Display results
print("\n" + "="*50)
print("Unlearning Results:")
print("="*50)
print(f"Method: {results['method']}")
print(f"Iterations: {results['iterations']}")
print(f"\nInitial Metrics:")
print(f"  Forget Accuracy: {results['initial_metrics']['forget_accuracy']:.4f}")
print(f"  Retain Accuracy: {results['initial_metrics']['retain_accuracy']:.4f}")
print(f"\nFinal Metrics:")
print(f"  Forget Accuracy: {results['final_metrics']['forget_accuracy']:.4f}")
print(f"  Retain Accuracy: {results['final_metrics']['retain_accuracy']:.4f}")
print(f"\nForget Quality: {results['forget_quality']:.4f}")
print(f"Utility Preservation: {results['utility_preservation']:.4f}")

## 4. Influence Function Computation

In [ ]:
# Create influence computer
influence_computer = create_influence_computer(
    method="lissa",
    lissa_iterations=50
)

# Get a test sample
test_data, test_target = next(iter(test_loader))
test_sample = test_data[0:1].to(device)
test_label = test_target[0:1].to(device)

# Compute influences
print("Computing influence scores...")
influences = influence_computer.compute_influence_lissa(
    model, test_sample, test_label, train_loader, device, iterations=50
)

# Display influence magnitudes
print("\nInfluence Scores by Layer:")
for name, influence in influences.items():
    magnitude = torch.norm(influence).item()
    print(f"  {name}: {magnitude:.6f}")

print("\n✅ Influence computation complete!")

## 5. Privacy Evaluation with Membership Inference Attack

In [ ]:
# Create member and non-member data loaders
member_loader = create_synthetic_dataset(200)  # Training data
non_member_loader = create_synthetic_dataset(200)  # Holdout data

# Initialize MIA
mia = MembershipInferenceAttack()

# Prepare attack data
print("Preparing membership inference attack...")
features, labels = mia.prepare_attack_data(
    model, member_loader, non_member_loader, device
)

# Train attack model
attack_train_acc = mia.train_attack_model(features, labels)
print(f"Attack model training accuracy: {attack_train_acc:.4f}")

# Evaluate attack
attack_result = mia.evaluate_attack(
    model, member_loader, non_member_loader, device
)

print("\n" + "="*50)
print("Privacy Evaluation Results:")
print("="*50)
print(f"Attack Accuracy: {attack_result.attack_accuracy:.4f}")
print(f"Attack AUC: {attack_result.attack_auc:.4f}")
print(f"Privacy Leakage: {(attack_result.attack_accuracy - 0.5) * 100:.2f}%")
print(f"\nNote: Attack accuracy close to 50% (random) indicates strong privacy!")

## 6. Putting It All Together: Complete Pipeline

In [ ]:
def run_complete_privacy_pipeline():
    """
    Demonstrates the complete privacy-preserving ML workflow.
    """
    print("\n" + "="*70)
    print("COMPLETE PRIVACY-PRESERVING ML PIPELINE")
    print("="*70)
    
    # Step 1: Train model
    print("\n[Step 1] Training model...")
    model = SimpleClassifier().to(device)
    # ... (training code from above)
    
    # Step 2: Evaluate initial privacy
    print("\n[Step 2] Initial privacy evaluation...")
    # ... (MIA code from above)
    
    # Step 3: Perform unlearning
    print("\n[Step 3] Unlearning sensitive data...")
    # ... (unlearning code from above)
    
    # Step 4: Re-evaluate privacy
    print("\n[Step 4] Post-unlearning privacy evaluation...")
    # ... (MIA code again)
    
    # Step 5: Compute influences
    print("\n[Step 5] Analyzing data influence...")
    # ... (influence code from above)
    
    print("\n✅ Complete pipeline executed successfully!")

# Uncomment to run:
# run_complete_privacy_pipeline()

## Next Steps

Explore more advanced tutorials:

1. **02_advanced_unlearning.ipynb** - Advanced unlearning techniques
2. **03_differential_privacy.ipynb** - Differential privacy training
3. **04_influence_functions_deep_dive.ipynb** - In-depth influence analysis
4. **05_face_recognition_example.ipynb** - Real-world face recognition application
5. **06_health_prediction_example.ipynb** - Healthcare AI with privacy

## Resources

- **Documentation**: Check the `docs/` directory
- **Examples**: See `examples/` directory
- **API Reference**: Run `help(module_name)` for any module

Happy privacy-preserving machine learning! 🔒🤖